# Installing and importing of required modules:

In [2]:
%pip install -q amplpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.0 MB/s eta 0:00:00


In [3]:
from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(modules = ["cbc"], license_uuid = "default")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).


In [4]:
import pandas as pd

# Writing the model and (simulated) data file:

In [5]:
%%writefile multicommodity.mod
param num_warehouses > 0;
param num_customers > 0;

set WAREHOUSES := 1..num_warehouses;
set CUSTOMERS  := 1..num_customers;

set COMMODITIES; # e.g., Electronics, Apparel, Groceries

param wh_x {WAREHOUSES};
param wh_y {WAREHOUSES};
param cust_x {CUSTOMERS};
param cust_y {CUSTOMERS};

param capacity {WAREHOUSES} > 0; # Shared volumetric capacity limit
param fixed_cost {WAREHOUSES} > 0;

param demand {CUSTOMERS, COMMODITIES} >= 0 integer; # Specific demand per customer per product type

param distance {w in WAREHOUSES, c in CUSTOMERS}
    := sqrt((wh_x[w] - cust_x[c])^2 + (wh_y[w] - cust_y[c])^2);

param cost_per_km := 1.5;
param ship_cost {w in WAREHOUSES, c in CUSTOMERS} := distance[w,c] * cost_per_km;

# DECISION VARIABLES
var Ship {WAREHOUSES, CUSTOMERS, COMMODITIES} >= 0 integer;
var Open {WAREHOUSES} binary;

# OBJECTIVE FUNCTION
minimize Total_Network_Cost:
    sum {w in WAREHOUSES, c in CUSTOMERS, k in COMMODITIES} ship_cost[w,c] * Ship[w,c,k] +
    sum {w in WAREHOUSES} fixed_cost[w] * Open[w];

# CONSTRAINTS

# 1. Volumetric Capacity Constraint (SUMMING OVER ALL COMMODITIES)
subject to Warehouse_Capacity {w in WAREHOUSES}:
    sum {c in CUSTOMERS, k in COMMODITIES} Ship[w,c,k] <= capacity[w] * Open[w];

# 2. Customer Demand Constraint (Enforced per specific product)
subject to Customer_Demand {c in CUSTOMERS, k in COMMODITIES}:
    sum {w in WAREHOUSES} Ship[w,c,k] >= demand[c,k];

Writing multicommodity.mod


In [6]:
%%writefile multicommodity.dat
param num_warehouses := 5;
param num_customers := 50;

set COMMODITIES := Electronics Apparel Groceries;

let {w in WAREHOUSES} wh_x[w] := Uniform(10, 90);
let {w in WAREHOUSES} wh_y[w] := Uniform(10, 90);
let {c in CUSTOMERS} cust_x[c] := Uniform(0, 100);
let {c in CUSTOMERS} cust_y[c] := Uniform(0, 100);

let {w in WAREHOUSES} capacity[w] := floor(1500 + 1000*Uniform01());
let {w in WAREHOUSES} fixed_cost[w] := Uniform(8000, 12000);

let {c in CUSTOMERS} demand[c, 'Electronics'] := floor(5 + 15*Uniform01());
let {c in CUSTOMERS} demand[c, 'Apparel']     := floor(15 + 20*Uniform01());
let {c in CUSTOMERS} demand[c, 'Groceries']   := floor(30 + 30*Uniform01());

Writing multicommodity.dat


#Solving the model and displaying/exporting the output:

In [8]:
ampl.read("multicommodity.mod")
ampl.read_data("multicommodity.dat")
ampl.set_option("solver", "cbc")
ampl.solve()

flat_ship_df = ampl.get_variable("Ship").to_pandas().reset_index()

# Rename the columns to make them intuitive before manipulation
flat_ship_df.columns = ["Warehouse", "Customer", "Commodity", "Volume"]

# 2. Export each commodity's routing matrix to its own tab in Excel
with pd.ExcelWriter("Multi_Commodity_Logistics_Plan.xlsx", engine="openpyxl") as writer:

    # Loop through each unique product category present in the data
    for commodity_name in flat_ship_df["Commodity"].unique():

        # Filter rows belonging only to the current product
        sub_df = flat_ship_df[flat_ship_df["Commodity"] == commodity_name]

        # Pivot the data: Rows = Customers, Columns = Warehouses
        matrix_df = sub_df.pivot(index="Customer", columns="Warehouse", values="Volume").fillna(0)

        # Universal cleanup: strip axis metadata labels for a pristine appearance
        matrix_df.index.names = [None]
        matrix_df.columns.names = [None]

        # Write this specific product's matrix to its own Excel Sheet
        matrix_df.to_excel(writer, sheet_name=str(commodity_name))

print("Multi-commodity optimization complete! Check your file sidebar for 'Multi_Commodity_Logistics_Plan.xlsx'.")

cbc 2.10.12: cbc 2.10.12: optimal solution; objective 173168.3758
122 simplex iterations
122 barrier iterations
Multi-commodity optimization complete! Check your file sidebar for 'Multi_Commodity_Logistics_Plan.xlsx'.


In [10]:
ampl.reset()

In [11]:
ampl.read("multicommodity.mod")
ampl.read_data("multicommodity.dat")
ampl.set_option("solver", "cbc")

# Define the demand scale multipliers we want to test (e.g., Normal, Peak, Extreme)
scenarios = {
    "Normal_Demand": 1.0,
    "Peak_Season_120pct": 1.2,
    "Extreme_Crisis_150pct": 1.5
}

# Create an Excel Writer engine to save multiple sheets to one file
with pd.ExcelWriter("Enterprise_Logistics_Scenarios.xlsx", engine="openpyxl") as writer:

    for name, multiplier in scenarios.items():
        print(f"Running Optimization Scenario: {name}...")

        # 1. Access the demand parameter directly inside AMPL and scale it
        # We fetch the values, scale them mathematically via Python, and re-inject them
        base_demands = ampl.get_parameter("demand").get_values().to_dict()
        scaled_demands = {c: int(val * multiplier) for c, val in base_demands.items()}
        ampl.get_parameter("demand").set_values(scaled_demands)

        # 2. Run the solver for this specific scenario iteration
        ampl.solve()

        # 3. Pull out the binary open status to see which warehouses survived selection
        open_df = ampl.get_variable("Open").to_pandas()
        print(f"Activated Warehouses for {name}:")
        #print(open_df.keys())
        print(open_df[open_df['Open.val'] > 0.5].index.tolist())

        # 4. Extract and Pivot the Shipping Matrix Data
        flat_ship_df = ampl.get_variable("Ship").to_pandas()
        matrix_df = flat_ship_df.unstack(level=1).fillna(0)

        # 5. Apply our universal structural cleanup to remove header artifacts
        matrix_df.columns = [f"{str(c[1])}" for c in matrix_df.columns.values]
        matrix_df.index.names = [None, None]

        # 6. Push this specific dataframe directly into its own unique Excel tab
        matrix_df.to_excel(writer, sheet_name=name)

print("\nSuccess! Open the file explorer on the left sidebar to download 'Enterprise_Logistics_Scenarios.xlsx'.")

Running Optimization Scenario: Normal_Demand...
cbc 2.10.12: cbc 2.10.12: optimal solution; objective 173168.3758
122 simplex iterations
122 barrier iterations
Activated Warehouses for Normal_Demand:
[1, 2, 3]
Running Optimization Scenario: Peak_Season_120pct...
cbc 2.10.12: cbc 2.10.12: optimal solution; objective 201081.1858
92 simplex iterations
92 barrier iterations
Activated Warehouses for Peak_Season_120pct:
[1, 2, 3]
Running Optimization Scenario: Extreme_Crisis_150pct...
cbc 2.10.12: cbc 2.10.12: optimal solution; objective 293233.0115
170 simplex iterations
170 barrier iterations
2 branching nodes
Activated Warehouses for Extreme_Crisis_150pct:
[1, 2, 3, 5]

Success! Open the file explorer on the left sidebar to download 'Enterprise_Logistics_Scenarios.xlsx'.


In [12]:
import folium
from folium import plugins

# 1. Fetch current coordinates and decisions from the active AMPL model instance
wh_x = ampl.get_parameter("wh_x").get_values().to_dict()
wh_y = ampl.get_parameter("wh_y").get_values().to_dict()
cust_x = ampl.get_parameter("cust_x").get_values().to_dict()
cust_y = ampl.get_parameter("cust_y").get_values().to_dict()

capa = ampl.get_parameter("capacity").get_values().to_dict()
dema = ampl.get_parameter("demand").get_values().to_dict()

commo = ampl.get_set("COMMODITIES").get_values().to_dict()

open_status = ampl.get_variable("Open").to_dict()
ship_plan = ampl.get_variable("Ship").to_dict()

# Anchor map near Mumbai (scaling your 0-100 grid into fractional lat/long shifts)
lat_center, lon_center = 19.0760, 72.8777
m = folium.Map(location=[lat_center, lon_center], zoom_start=8, tiles=None)

folium.TileLayer(
    tiles="https://wikimedia.org{z}/{x}/{y}.png",
    attr='&copy; <a href="https://wikimediafoundation.org">Wikimedia</a>',
    name="Wikimedia Clean Map"
).add_to(m)

# 2. Plot Warehouses (Blue if Active, Red if Closed by solver)
for w in wh_x.keys():
    # Transform grid values to slight offsets around Mumbai
    w_lat = lat_center + (wh_y[w] - 50) / 100.0
    w_lon = lon_center + (wh_x[w] - 50) / 100.0

    is_open = open_status[w] > 0.5
    color = "blue" if is_open else "red"
    icon_type = "cloud" if is_open else "info-sign"

    folium.Marker(
        location=[w_lat, w_lon],
        popup=f"Warehouse {int(w)} | Status: {'OPEN' if is_open else 'CLOSED'}",
        icon=folium.Icon(color=color, icon=icon_type)
    ).add_to(m)

# 3. Plot Customers and active logistics paths
for wc, volume in ship_plan.items():
    if volume > 0.1:  # Only map active routing assignments
        w,c,p = wc
        w_lat = lat_center + (wh_y[w] - 50) / 100.0
        w_lon = lon_center + (wh_x[w] - 50) / 100.0
        c_lat = lat_center + (cust_y[c] - 50) / 100.0
        c_lon = lon_center + (cust_x[c] - 50) / 100.0

        # Plot customer node

        folium.CircleMarker(
            location=[c_lat, c_lon],
            radius=8,                 # Marker size
            color="#FFFFFF",          # Outer border color (White creates a sharp drop-off)
            weight=2.0,               # Border thickness in pixels
            opacity=1.0,              # Solid border crispness
            fill=True,
            fill_color="#EF4444",     # Core inner color (e.g., Crimson Red)
            fill_opacity=1.0,         # 100% solid fill stops background bleed-through
            tooltip=f"Customer {int(c)}"
        ).add_to(m)

        # Draw dynamic freight flow line (thickness proportional to cargo volume)
        folium.PolyLine(
            locations=[[w_lat, w_lon], [c_lat, c_lon]],
            weight=max(1, volume / 15.0),  # Adjust scaling denominator based on demand values
            color="indigo",
            opacity=0.6,
            smooth_factor=1.0,        # Maximizes crisp vector resolution
            line_cap="round",         # Prevents jagged, blocky edge terminations
            popup=f"Flow: {volume:.1f} units"
        ).add_to(m)

# Save the final file map to download or display
m.save("Logistics_Network_Flow_Map.html")
print("Map generated! Download 'Logistics_Network_Flow_Map.html' from your file sidebar and double-click to view.")

m

Map generated! Download 'Logistics_Network_Flow_Map.html' from your file sidebar and double-click to view.


In [13]:
print(commo)
print(dema[(1,'Apparel')])

{'Electronics': None, 'Apparel': None, 'Groceries': None}
27


In [17]:
import folium, random
from folium import plugins # Needed for AntPath arrow animations

lat, lon = 19.0760, 72.8777

# 1. Initialize map and base layers
m = folium.Map(location=[lat, lon], zoom_start=10, tiles=None)
folium.TileLayer(tiles="https://wikimedia.org{z}/{x}/{y}.png", attr='&copy; Wikimedia', name="Base Map").add_to(m)

# 2. Create feature groups for the toggle panel
warehouse_group = folium.FeatureGroup(name="🏭 Warehouses").add_to(m)
customer_group = folium.FeatureGroup(name="👥 Customers").add_to(m)
#arc_group = folium.FeatureGroup(name="🚚 Logistics Flows").add_to(m)

commo_colors = dict((i, f"#{random.randint(0, 0xFFFFFF):06x}") for i in commo.keys())
layers = {}
for commodity in commo.keys():
    layers[commodity] = folium.FeatureGroup(name=f"📦 {commodity} Flow").add_to(m)

# --- WAREHOUSE LOOP ---
for w in wh_x.keys():
    # Transform grid values to slight offsets around Mumbai
    w_lat = lat_center + (wh_y[w] - 50) / 100.0
    w_lon = lon_center + (wh_x[w] - 50) / 100.0

    is_open = open_status[w] > 0.5
    color = "blue" if is_open else "red"
    icon_type = "cloud" if is_open else "info-sign"

    folium.Marker(
        location=[w_lat, w_lon],
        popup=f"Warehouse: {int(w)}<br><br>Status: {'OPEN' if is_open else 'CLOSED'}<br><br>Capacity: {capa[w]}",
        icon=folium.Icon(color=color, icon=icon_type)
    ).add_to(m)

# --- CUSTOMER LOOP ---
for cust in cust_x.keys():
    dema_cust = '<div style="padding-left: 40px;"><table><tbody>'+''.join(['<tr><th><b>'+str(i)+':</b></th><td>'+str(dema[(cust,i)])+'</td></tr>' for i in commo])+'</tbody></table></div>'
    c_lat = lat + (cust_y[cust] - 50) / 100.0
    c_lon = lon + (cust_x[cust] - 50) / 100.0
    folium.CircleMarker(
        location=[c_lat, c_lon],
        radius=6,                # Smaller footprint
        color="#FFFFFF",         # White border to separate from background
        weight=1.5,
        fill=True,
        fill_color="#EF4444",    # Vibrant red for demand endpoints
        fill_opacity=0.9,
        tooltip=f"<b>Customer:</b> {cust}<br><br>{dema_cust}"
    ).add_to(customer_group)

# --- ARCS/FLOWS LOOP ---
for arc, volume in ship_plan.items():
    w,c,p = arc

    if volume:
    # UPGRADE: Animated Directional Flow instead of static line
        w_lat = lat + (wh_y[w] - 50) / 100.0
        w_lon = lon + (wh_x[w] - 50) / 100.0
        c_lat = lat + (cust_y[c] - 50) / 100.0
        c_lon = lon + (cust_x[c] - 50) / 100.0
        plugins.AntPath(
            locations=[[w_lat, w_lon], [c_lat, c_lon]],
            weight=5,
            opacity=0.8,
            color=commo_colors[p],
            pulse_color="#FFFFFF",   # Color of the moving "ants" indicating direction
            delay=1200,              # Speed of animation (lower is faster)
            dash_array=[10, 20],     # Length and spacing of the animated segments
            tooltip=f"<b>Route:</b> Warehouse: {w} ➔ Customer: {c}<br><b>Flow Vol:</b> {round(volume)}"
        ).add_to(layers[p])

# 3. Add Layer Control to the top-right corner
folium.LayerControl(collapsed=False).add_to(m)

m.save("Multicommodity_Network_Flow_Map.html")
# 4. Render
m
